### Parameter Efficient Fine-Tuning
In this notebook, you're gonna fine-tune large language models within limited GPU memory.

In [1]:
# Original library versions
# %pip install --quiet transformers==4.34.1 accelerate==0.24.0 sentencepiece==0.1.99 optimum==1.13.2 peft==0.5.0 bitsandbytes==0.41.2.post2

# Preferred versions for Colab as of October 2025 (thanks, Lev!)
# %pip install --quiet "bitsandbytes==0.45.3" "transformers>=4.43,<4.46" "accelerate>=0.33,<0.36" "peft>=0.11.1" "optimum>=1.20.0" "sentencepiece"

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import transformers
from tqdm.auto import tqdm
assert torch.cuda.is_available(), "you need cuda for this part"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
def reload_model(model_name = 'Enoch/llama-7b-hf'):
    # loading Llama tokenizer ...
    tokenizer = transformers.LlamaTokenizer.from_pretrained(model_name, device_map=device)
    tokenizer.pad_token_id = tokenizer.eos_token_id

    # ... and the model itself
    model = transformers.AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map='auto',
        low_cpu_mem_usage=True,
        offload_state_dict=True,
        load_in_4bit=True,
        torch_dtype=torch.float32,  # weights are 4-bit; layernorms and activations are fp32
    )
    for param in model.parameters():
        param.requires_grad=False

    model.gradient_checkpointing_enable()  # only store a small subset of activations, re-compute the rest.
    model.enable_input_require_grads()     # override an implementation quirk in gradient checkpoints that disables backprop unless inputs require grad
    # more on gradient checkpointing: https://pytorch.org/docs/stable/checkpoint.html https://arxiv.org/abs/1604.06174
    return model, tokenizer

In [4]:
model, tokenizer = reload_model()

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/33 [00:00<?, ?it/s]

### Prompt tuning: the story of a fox (1 point)

![img](https://i.imgur.com/Ux3qQAu.png) (source: theodd1souts.fandom.com)

In [5]:
prompt = 'A quick brown fox'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)

for i in range(10):
    next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
    batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
    batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=-1)

print("\nOutput:", tokenizer.decode(batch['input_ids'][0].cpu().numpy().tolist()))

Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)



Output: <s>A quick brown fox jumps over the lazy dog.
A quick


What a blatant lie! This particular fox assures you that it didn't in fact jump over the lazy dog. No, sir! The fox was just minding its own business. __Your task is to train the model to say truth: no dog was jumped over today.__

In [6]:
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
outputs = model(**batch)

next_word_logits = outputs.logits[:, :-1]
true_next_tokens = batch['input_ids'][:, 1:]
loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))

print("Loss:", loss)

Loss: tensor(3.0729, device='cuda:0', grad_fn=<NllLossBackward0>)


Except, we can't train the entire model - that would be 28GB gradients in float32. Instead, let's run [prompt tuning](https://arxiv.org/abs/2104.08691).

![img](https://i.imgur.com/VwNNKnb.png)


In [7]:
class WordEmbeddingsWithLearnedPrompts(nn.Module):
    """
    To perform prompt tuning, you will need to replace the model's original word embeddings with a layer - THIS layer
    - that inserts trainable prompts instead of the first N token embeddings.
    """

    def __init__(self, word_embeddings: nn.Embedding, num_prompts: int):
        super().__init__()
        self.original_word_embeddings = word_embeddings
        self.num_prompts = num_prompts
        self.learnable_prompts = nn.Parameter(
            torch.randn(1, num_prompts, word_embeddings.embedding_dim), requires_grad=True
        )

    def forward(self, input_ids: torch.LongTensor):
        # input_ids shape: [batch_size, seq_length]
        assert input_ids.dtype == torch.int64
        assert input_ids.shape[1] > self.num_prompts
        assert torch.all(input_ids[:, :self.num_prompts] == tokenizer.pad_token_id).item(), (
            "Don't forget to prepend several BOS tokens to input_ids"
        )

        # Embed the input_ids using the original word embeddings
        input_embeddings = self.original_word_embeddings(input_ids)  # Shape: [batch_size, seq_length, embedding_dim]

        # Replace the first num_prompts token embeddings with the learnable prompts
        batch_size = input_ids.shape[0]
        learnable_prompts_expanded = self.learnable_prompts.expand(batch_size, -1, -1)  # Shape: [batch_size, num_prompts, embedding_dim]
        remaining_embeddings = input_embeddings[:, self.num_prompts:, :]  # Shape: [batch_size, seq_length - num_prompts, embedding_dim]

        # Concatenate learnable prompts with the embeddings of the remaining tokens
        output_embeddings = torch.cat([learnable_prompts_expanded, remaining_embeddings], dim=1)  # Shape: [batch_size, seq_length, embedding_dim]

        return output_embeddings


In [8]:
num_prompts = 16
test_emb_layer = WordEmbeddingsWithLearnedPrompts(model.model.embed_tokens, num_prompts=num_prompts).to(device)
test_input_ids = tokenizer("a cat say on a may", return_tensors='pt')['input_ids'].to(device)

space_for_prompts = torch.full([len(test_input_ids), num_prompts], fill_value=tokenizer.pad_token_id,
                               dtype=torch.int64, device=device)
test_inputs_with_prompts = torch.cat([space_for_prompts, test_input_ids], dim=1)

with torch.cuda.amp.autocast():
  test_prompt_embeddings = test_emb_layer(test_inputs_with_prompts)

assert test_prompt_embeddings.shape[:2] == test_inputs_with_prompts.shape
assert test_prompt_embeddings.shape[-1] == model.config.hidden_size
assert torch.allclose(test_prompt_embeddings[:, :num_prompts], test_emb_layer.learnable_prompts.float())
assert torch.allclose(test_prompt_embeddings[:, num_prompts:], model.model.embed_tokens(test_input_ids).float())
print("Looks legit!")

Looks legit!


/tmp/ipykernel_132595/1152930240.py:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


__Now that it works,__ let's inject learnable prompts into the main model and teach it about foxes.

In [ ]:
model, tokenizer = reload_model()

assert isinstance(model.model.embed_tokens, nn.Embedding), "you have already replaced the embedding layer. If the replacement is broken, please reload the model"

model.model.embed_tokens = WordEmbeddingsWithLearnedPrompts(model.model.embed_tokens, num_prompts=num_prompts).to(device)

opt = torch.optim.Adam([model.model.embed_tokens.learnable_prompts], lr=0.01)

In [9]:
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
space_for_prompts = torch.full([len(test_input_ids), num_prompts], fill_value=tokenizer.pad_token_id,
                               dtype=torch.int64, device=device)
batch['input_ids'] = torch.cat([space_for_prompts, batch['input_ids']], dim=1)
batch['attention_mask'] = torch.cat([torch.ones_like(space_for_prompts), batch['attention_mask']], dim=1)

model.train()

with tqdm(range(100), desc="Training soft prompts") as pbar: 
    for i in pbar:
        model.zero_grad()
        outputs = model(**batch)
        next_word_logits = outputs.logits[:, num_prompts : -1, :]
        true_next_tokens = batch['input_ids'][:, num_prompts + 1:]
        loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))
        pbar.set_postfix(loss=loss.item())
        loss.backward()
        opt.step()

model.eval()

# raise NotImplemented("Your task: iteratively train the model to reduce loss using prompt optimizer (opt)")

Training soft prompts:   0%|          | 0/100 [00:00<?, ?it/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): WordEmbeddingsWithLearnedPrompts(
      (original_word_embeddings): Embedding(32000, 4096, padding_idx=0)
    )
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear4bit(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaR

In [10]:
# Final loss assertion
assert loss.item() <= 0.1
print("Good job!")

Good job!


In [11]:
prompt = 'A quick brown fox'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
batch['input_ids'] = torch.cat([space_for_prompts, batch['input_ids']], dim=1)
batch['attention_mask'] = torch.cat([torch.ones_like(space_for_prompts), batch['attention_mask']], dim=1)


for i in range(15):
    next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
    batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
    batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=-1)

print("\nOutput:", tokenizer.decode(batch['input_ids'][0, num_prompts:].cpu().numpy().tolist()))

# if you did everything right, the model will deny that the fox jumped over the lazy dog


Output: <s>A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it


### Using HuggingFace PEFT (2 point)

[`peft`](https://huggingface.co/docs/peft/index) is a transformer's sister library that allows you to apply various __p__arameter __e__fficient __f__ine-__t__uning methods to pre-trained transformers. The library imlements both prompt tuning, prefix tuning, as well as several adapter-based techniques under a common interface:



In [9]:
import peft
assert isinstance(model.model.embed_tokens, nn.Embedding), "please reload the model"

peft_config = peft.PromptTuningConfig(task_type=peft.TaskType.CAUSAL_LM, num_virtual_tokens=16)
model = peft.get_peft_model(model, peft_config).to(device)  # note: for most peft methods, this line also modifies model in-place
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Total parameters (excluding quantization):", sum(p.numel() for p in model.parameters()))

Trainable parameters: 65536
Total parameters (excluding quantization): 3500478464


In [10]:
# Your task: optimize the PEFT-wrapped model to achieve next token prediction loss < 0.1, but this time using PEFT
# Please note: you no longer need to prepend PAD tokens, but you still need to skip :num_virtual_tokens: first logits.
# Finally, generate the sentence to make sure that the model learned the truth.

In [11]:
# Define the ground truth sentence
# the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors="pt", return_token_type_ids=False).to(device)

In [12]:
# Training Configuration
loss_threshold = 0.1  # Desired loss threshold
num_epochs = 200  # Max number of epochs
learning_rate = 0.3 # as per original paper #<YOUR CODE HERE>  # Learning rate

# Define the optimizer for trainable parameters (PEFT prompts)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Define the ground truth
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors="pt", return_token_type_ids=False).to(device)

# Training Loop
for epoch in range(num_epochs):
    # Forward pass
    outputs = model(**batch)  #<YOUR CODE HERE>

    # Skip logits for virtual tokens and the last token
    next_word_logits = outputs.logits[:, peft_config.num_virtual_tokens : -1, :]  # Skip virtual tokens
    true_next_tokens = batch['input_ids'][:, 1:]  # Shift ground truth tokens by one

    # Compute the loss
    loss = F.cross_entropy(
        next_word_logits.flatten(0, 1),
        true_next_tokens.flatten(0, 1)
    )

    # Backpropagation
    loss.backward()
    optimizer.step()

    # Print loss for tracking
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {loss.item()}")

    # Stop training if loss is below threshold
    if loss.item() < loss_threshold:
        print("Loss threshold reached. Stopping training.")
        break
else:
    print("Maximum epochs reached without meeting the loss threshold.")

Epoch 1/200, Loss: 7.428136825561523
Epoch 2/200, Loss: 4.847543716430664
Epoch 3/200, Loss: 3.7659246921539307
Epoch 4/200, Loss: 5.723677158355713
Epoch 5/200, Loss: 4.7074503898620605
Epoch 6/200, Loss: 3.0984487533569336
Epoch 7/200, Loss: 2.1482534408569336
Epoch 8/200, Loss: 4.4446587562561035
Epoch 9/200, Loss: 5.009557723999023
Epoch 10/200, Loss: 3.184415340423584
Epoch 11/200, Loss: 2.4332005977630615
Epoch 12/200, Loss: 2.2021102905273438
Epoch 13/200, Loss: 2.1683833599090576
Epoch 14/200, Loss: 2.176722764968872
Epoch 15/200, Loss: 2.2411341667175293
Epoch 16/200, Loss: 2.2712666988372803
Epoch 17/200, Loss: 2.2696540355682373
Epoch 18/200, Loss: 2.242471218109131
Epoch 19/200, Loss: 2.192744731903076
Epoch 20/200, Loss: 2.1320600509643555
Epoch 21/200, Loss: 2.062617540359497
Epoch 22/200, Loss: 1.9796738624572754
Epoch 23/200, Loss: 1.8906452655792236
Epoch 24/200, Loss: 1.8218445777893066
Epoch 25/200, Loss: 1.7739837169647217
Epoch 26/200, Loss: 1.7332217693328857
Epoc

In [13]:
# Final assertion to ensure loss is below threshold
assert loss.item() < loss_threshold, "Training failed to reduce loss below threshold."
print("Training successful! Loss is below 0.1.")

Training successful! Loss is below 0.1.


In [14]:
prompt = "A quick brown fox"
batch = tokenizer(prompt, return_tensors="pt", return_token_type_ids=False).to(device)

# Generate 18 tokens
for i in range(15):
    # Forward pass to get the logits
    outputs = model(**batch)
    next_token = outputs.logits[0, -1].argmax(-1).reshape(1, 1)

    # Append the next token to input_ids
    batch["input_ids"] = torch.cat([batch["input_ids"], next_token], dim=-1)

    # Update the attention_mask to match the new input_ids length
    new_attention_mask = torch.ones_like(next_token, dtype=batch["attention_mask"].dtype).to(device)
    batch["attention_mask"] = torch.cat([batch["attention_mask"], new_attention_mask], dim=-1)

# Decode the generated sequence
# Skip the virtual tokens (if applicable) by slicing `batch["input_ids"][:, num_prompts:]`
decoded_output = tokenizer.decode(batch["input_ids"][0].cpu().numpy().tolist(), skip_special_tokens=True)
print("\nOutput:", decoded_output)



Output: A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it


### Parameter-efficient finetuning with LoRA (2 points)

When training on more serious tasks, you can use low-rank adapters based on the [LoRA paper](https://arxiv.org/pdf/2106.09685.pdf).

The core idea is to add low-rank adapters __in parallel with existing linear layers,__ like this:
<center><img src="https://i.imgur.com/6bQLNiG.png" width=240px></center>

In the original LoRA paper, the adapters were only added to attention projection matrices. However, [subsequent works](https://arxiv.org/abs/2305.14314) show that it is useful to adapt FFNs as well. But before we do any training, we need to implement the basic LoRA layer.

In [15]:
# re-load the model to remove any previous PEFT tuners
model, tokenizer = reload_model()

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/33 [00:00<?, ?it/s]

In [16]:
class LoRALayer(nn.Module):
    """Wraps a linear layer with LoRA-like adapter. Wraps an existing OPT linear layer"""
    def __init__(self, module: nn.Linear, rank: int):
        super().__init__()
        self.module = module  # pre-trained (frozen) linear layer
        self.adapter_A = nn.Parameter(torch.empty(module.in_features, rank, device=module.weight.device))
        nn.init.kaiming_uniform_(self.adapter_A, a=5 ** 0.5)
        self.adapter_B = nn.Parameter(torch.zeros(rank, module.out_features, device=module.weight.device))

    def forward(self, input):
        # Apply self.module and LoRA adapter, return the sum (self.module outputs + adapter outputs)
        original_output = self.module(input) # <YOUR CODE HERE>
        lora_output = input @ self.adapter_A @ self.adapter_B # <YOUR CODE HERE>

        return original_output + lora_output

In [17]:
# test your implementation
test_linear = nn.Linear(128, 128)
test_linear.weight.data[...] = torch.eye(128)
test_adapter = LoRALayer(test_linear, rank=8)

assert torch.allclose(test_adapter(torch.ones(1, 1, 128)), test_linear.bias + 1), "please check your forward pass"

test_adapter.adapter_A.data[...] = torch.linspace(0.1, -0.5, 128 * 8).view(128, 8)
test_adapter.adapter_B.data[...] = torch.linspace(0.5, -0.1, 128 * 8).view(8, 128)
test_linear.bias.data[...] = torch.linspace(1., -1., 128)

dummy_loss = F.mse_loss(test_adapter(torch.ones(1, 128) / 128).squeeze(), torch.linspace(-1, 1, 128))
assert torch.allclose(dummy_loss, torch.tensor(1.3711389), rtol=0, atol=1e-4)
dummy_loss.backward()
assert all(w.grad is not None for w in [test_adapter.adapter_A, test_adapter.adapter_B]), "some adapter weights have no grad"
assert torch.allclose(test_adapter.adapter_A.grad.sum(), torch.tensor(-0.60158), rtol=0, atol=1e-4), "bad grad w.r.t. A"
assert torch.allclose(test_adapter.adapter_B.grad.sum(), torch.tensor(0.9931), rtol=0, atol=1e-4), "bad grad w.r.t. B"
# note: bad grad means that your code is different from LoRA paper OR that your code is not autograd-friendly (e.g. no_grad)
del dummy_loss, test_linear, test_adapter
print("All tests passed!")

All tests passed!


### Apply LoRA to the model

The code below applies LoRA adapters on top of Q/K/V linear layers in Llama attention. You may also choose to modify other layers:
* self_attn.o_proj - attention output projection
* mlp.up_proj, mlp.gate_proj, mlp.down_proj - transformer feedforward layers
* lm_head - output LM head

In [18]:
lora_rank = 8

for name, module in model.model.layers.named_modules():
    if 'LlamaDecoderLayer' in repr(type(module)):
        module.self_attn.q_proj = LoRALayer(module.self_attn.q_proj, rank=lora_rank).to(device)
        module.self_attn.k_proj = LoRALayer(module.self_attn.k_proj, rank=lora_rank).to(device)
        module.self_attn.v_proj = LoRALayer(module.self_attn.v_proj, rank=lora_rank).to(device)

assert sum(isinstance(module, LoRALayer) for module in model.modules()) == 96  # for Llama-7B

In [19]:
batch = tokenizer("This model wants to share its greatest secret:", return_tensors='pt', return_token_type_ids=False)
# test a single training step, make sure we get meaningful gradients
with torch.cuda.amp.autocast(dtype=torch.float32):
    out = model.forward(**batch)
    (out.logits.norm() / 100).backward()

for i, module in enumerate(model.modules()):
    if isinstance(module, LoRALayer):
        assert module.adapter_B.grad is not None
        assert module.adapter_B.grad.norm().item() > 0

model.zero_grad(set_to_none=True)
print("Grad check successful, well done!")

/tmp/ipykernel_132595/3590305689.py:3: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float32):


Grad check successful, well done!


### (example) How to train your model

The example below shows how to train the LoRA adapters on a dummy dataset. You will need to run a _similar_ training task later.

__Note:__ please scroll down for the homework task

In [ ]:
# checking if the model can learn. Change max_steps for proper training
# import datasets
# data = datasets.load_dataset("Abirate/english_quotes", split="train[:32]") # 32 lines
# data = data.map(lambda samples: tokenizer(samples['quote']), batched=True)
# model._hf_peft_config_loaded = True  # silence a warning from HF trainer

# trainer = transformers.Trainer(
#     model=model, train_dataset=data,
#     args=transformers.TrainingArguments(
#         per_device_train_batch_size=2, 
#         gradient_accumulation_steps=1, # for effectively larger batch size
#         warmup_steps=250, 
#         max_steps=100, 
#         learning_rate=2e-4, 
#         fp16=True,
#         logging_steps=1, 
#         output_dir='outputs', 
#         report_to=None,
#         save_strategy="no" # to make it work as of October 2025 (thanks, Vlad!) 
#     ),
#     data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
# )
# if you see cache warnings, set `model.config.use_cache = False` to silence them. Please re-enable for inference!

# trainer.train()

# NOTE: this is just an example! you do not have to wait for this progressbar to finish :)

### Final task: *actually* train the model (5 points)

Your task is to fine-tune the model to _generate python code_. Please use the above examples for inspiration. More specifically,

* __dataset:__ use [codeparrot-clean](https://huggingface.co/datasets/codeparrot/codeparrot-clean) or any other data containing python code. Since you do not need much data for this excercise, it is enough to use just shorter train subset of `codeparrots`
* __preprocessing:__ select python code based on file extentions (.py)  (may skip in case of codeparrot - it is 100% python)
* __short lines:__ please take the first 512 characters of each line
* __adapter type:__ please use LoRA as defined above __plus at least one of:__
   - extra adapter on lm_head
   - extra adapter on MLP components (mlp.*)
   - trainable input embeddings (requires tweaking memory usage)

* __training:__ you do not have to train to convergence. If all goes well, your model should `.generate` code after 500 steps. Please use batch size of at least 4 (4 x 1 x 512 tokens) using `gradient_accumulation_steps=4`.


Note: the peft library also has LoRA implementation. However, we ask that for this assignment you show at least one complete training run with your own LoRA code.

__Alternative assignment:__ Instead of doing python code, feel free to substitute the task with any other dataset, e.g. your favorite artist or podcast, as long as it's ethical. If you choose your own task, please show examples of what your model learned - or did not learn, akin to the code examples below.

In [45]:
prompts =  ['', 'import', 'from', 'while', 'try', 'if', 'for', 'torch']  # feel free to add a few more that are not 100% assiciated with Python

# <A WHOLE LOT OF YOUR CODE>
# generate baseline samples with the selected prompts before finetuning
# please feel free to use transformers.Trainer (as above) or your custom training code
# after the training concludes, please show examples of text generated by your model. It is expected to look like Python code fragments
# print the generation examples nicely (suggestion: use pandas or HTML) for easier comparison
# note: your LoRA-enhanced model can run generation the same way as the non-trained model (above)

In [ ]:
# Установка и импорт rich
# %pip install rich

from rich import print as rprint
from rich.console import Console
from rich.text import Text
from rich.panel import Panel
from rich.table import Table
from rich.columns import Columns
from rich.align import Align
from rich.syntax import Syntax

import torch
import torch.nn as nn
import torch.nn.functional as F

import datasets

import transformers
from tqdm.auto import tqdm
assert torch.cuda.is_available(), "you need cuda for this part"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Создание консоли для вывода
console = Console()

#### Re-load the model to remove any previous PEFT tuners

In [21]:
# re-load the model to remove any previous PEFT tuners
model, tokenizer = reload_model()

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/33 [00:00<?, ?it/s]

#### Load and preprocess the dataset

In [22]:
# Use first 1000 samples for faster training
dataset = datasets.load_dataset("codeparrot/codeparrot-clean", split="train[:1000]", cache_dir="./data")  

# Preprocess the data
def preprocess_code(example):
    # Take first 512 characters of each line
    code = example['content']
    lines = code.split('\n')
    processed_lines = [line[:512] for line in lines]
    return {'content': '\n'.join(processed_lines)}

# Apply preprocessing
dataset = dataset.map(preprocess_code)

# check the dataset
print(f"Dataset loaded: {len(dataset)} samples")
code_example = Syntax(dataset[0]['content'][700:1152] + "...", "python", theme="monokai", line_numbers=True)
console.print(Panel(code_example, title="Sample code", border_style="yellow"))


Resolving data files:   0%|          | 0/54 [00:00<?, ?it/s]

Dataset loaded: 1000 samples


╭────────────────────────────────────────────────── Sample code ──────────────────────────────────────────────────╮
│    1 #########################################################################                                  │
│    2                                                                                                            │
│    3 from autobahn.asyncio.websocket import WebSocketClientProtocol, \                                          │
│    4                                        WebSocketClientFactory                                              │
│    5                                                                                                            │
│    6 import asyncio                                                                                             │
│    7                                                                                                            │
│    8                                                                                                            │
│    9                                                                                                            │
│   10 class MyClientProtocol(WebSocketClientProtocol):                                                           │
│   11                                                                                                            │
│   12    def onConnect(self, response):                                                                          │
│   13       print("Server connected: {0}".format(response.peer))                                                 │
│   14                                                                                                            │
│   15    @asyncio.coroutine                                                                                      │
│   16    def onOpen(self):                                                                                       │
│   17       print("WebSocket connection open.")                                                                  │
│   18                                                                                                            │
│   19 ...                                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

#### Code samples BEFORE training

In [24]:
prompts = ['', 'import', 'from', 'while', 'try', 'if', 'for', 'torch', 'def', 'class']

def generate_code(model, prompt, max_length=100, temperature=0.7):
    """Generate code from a prompt"""
    model.eval()
    with torch.no_grad():
        # Tokenize
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        
        # Generate
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            num_return_sequences=1
        )
        
        # Decode
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        return generated_text

# Generate samples before training (baseline)
baseline_samples = {}
for prompt in tqdm(prompts, desc="Generating baseline samples", total=len(prompts)):
    baseline_samples[prompt] = generate_code(model,prompt, max_length=100)

Generating baseline samples:   0%|          | 0/10 [00:00<?, ?it/s]

In [25]:
# Print all samples as a single panel with two columns: Prompt and Sample
def print_samples(prompts, samples):
    table = Table(title="Baseline Samples (Before Training)", show_header=True, header_style="bold green", show_lines=True)
    table.add_column("Prompt", style="cyan", overflow="crop")
    table.add_column("Sample", style="green", overflow="crop")

    for prompt in prompts:
        table.add_row(
            repr(prompt),
            Syntax(samples.get(prompt, ""), "python", theme="monokai", line_numbers=True)
        )

    console.print(table)

print_samples(prompts, baseline_samples)
console.print("[bold green]Ну так себе программист[/bold green]")


                                        Baseline Samples (Before Training)                                         
┏━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Prompt   ┃ Sample                                                                                               ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ ''       │   1 ▲▼▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲ │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ 'import' │    1 import Foundation                                                                               │
│          │    2                                                                                                 │
│          │    3 internal protocol NSObject {                                                                    │
│          │    4                                                                                                 │
│          │    5 }                                                                                               │
│          │    6                                                                                                 │
│          │    7 extension NSObject: NSCopying {                                                                 │
│          │    8                                                                                                 │
│          │    9     internal func copy() -> NSObject {                                                          │
│          │   10         return self                                                                             │
│          │   11     }                                                                                           │
│          │   12 }                                                                                               │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ 'from'   │    1 from __future__ import print_function                                                           │
│          │    2                                                                                                 │
│          │    3 from collections import defaultdict                                                             │
│          │    4 from . import Dataset                                                                           │
│          │    5                                                                                                 │
│          │    6                                                                                                 │
│          │    7 class LatentDirichletAllocation(Dataset):                                                       │
│          │    8     r"""LatentDirichletAllocationDataset                                                        │
│          │    9                                                                                                 │
│          │   10     This dataset is a collection of documents in the form of a bag-of-words,                    │
│          │   11     where each document appears as a sequence of tokens. The vocabulary                         │
│          │   12     contains 200 unique tokens.                                                                 │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ 'while'  │   1 while I was there, I was able to do some shopping and buy some souvenirs for my friends and fami │
│          │   2 The people in the market are very friendly and are not pushy at all.            

Ну так себе программист

#### Add LoRA adapters to `self_attn`, `mlp` components and `lm_head`

In [26]:
assert 'LoRALayer' in globals(), "LoRALayer is not defined. Run the cell where LoRALayer is defined before proceeding."


# Add LoRA adapters to `self_attn`, `mlp` components and `lm_head`
lora_rank = 8
# Add LoRA 
for name, module in model.model.layers.named_modules():
    if 'LlamaDecoderLayer' in repr(type(module)):
        # Add LoRA to Q/K/V projections
        module.self_attn.q_proj = LoRALayer(module.self_attn.q_proj, rank=lora_rank).to(device)
        module.self_attn.k_proj = LoRALayer(module.self_attn.k_proj, rank=lora_rank).to(device)
        module.self_attn.v_proj = LoRALayer(module.self_attn.v_proj, rank=lora_rank).to(device)
        # Add LoRA to MLP components
        module.mlp.gate_proj = LoRALayer(module.mlp.gate_proj, rank=lora_rank).to(device)
        module.mlp.up_proj = LoRALayer(module.mlp.up_proj, rank=lora_rank).to(device)
        module.mlp.down_proj = LoRALayer(module.mlp.down_proj, rank=lora_rank).to(device)

# Add LoRA to lm_head
model.lm_head = LoRALayer(model.lm_head, rank=lora_rank).to(device)

# Count total LoRA layers
total_lora_layers = sum(isinstance(module, LoRALayer) for module in model.modules())
# Count trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
trainable_pct = 100 * trainable_params / total_params if total_params > 0 else 0.0

# Rich table for summary
lora_table = Table(title="Model LoRA and Parameter Summary", show_header=True, header_style="bold magenta")
lora_table.add_column("Metric", style="cyan", justify="left")
lora_table.add_column("Value", style="bold yellow", justify="right")

lora_table.add_row("Total LoRA Layers", f"{total_lora_layers}")
lora_table.add_row("Trainable Parameters", f"{trainable_params:,}")
lora_table.add_row("Total Parameters", f"{total_params:,}")
lora_table.add_row("Trainable Percentage", f"{trainable_pct:.2f}%")

console.print(lora_table)


    Model LoRA and Parameter Summary    
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Metric               ┃         Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Total LoRA Layers    │           193 │
│ Trainable Parameters │    18,180,096 │
│ Total Parameters     │ 3,518,593,024 │
│ Trainable Percentage │         0.52% │
└──────────────────────┴───────────────┘

#### Training using transformers.Trainer

In [27]:
# Prepare the dataset for Trainer
tokenized_dataset = dataset.map(
    lambda sample: tokenizer(sample["content"], truncation=True, padding=False, max_length=512),
    batched=True,
    remove_columns=dataset.column_names,
)


In [ ]:
# Set model config for training
model.config.use_cache = False  # Disable cache for training
model._hf_peft_config_loaded = True  # Silence warning from HF trainer

training_args = transformers.TrainingArguments(
    per_device_train_batch_size=1,  # Small batch size
    gradient_accumulation_steps=4,  # Effective batch size = 4
    warmup_steps=50,
    max_steps=500,                  # Train for 500 steps
    learning_rate=2e-4,
    fp16=True,                      # Use mixed precision
    logging_steps=10,
    output_dir='./outputs',
    report_to=None,
    save_strategy="no",             # Don't save checkpoints
    dataloader_drop_last=False,     # Don't drop last batch
)

# Create trainer
trainer = transformers.Trainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_args,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

trainer.train()

model.eval()
model.config.use_cache = True  # Re-enable cache for inference


/home/yc-user/jupyter-notebooks/NLP-course/6.Pretrain_finetune/.venv/lib/python3.12/site-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
max_steps is given, it will override any value given in num_train_epochs


Step,Training Loss
10,1.066500
20,0.918300
30,0.932600
40,0.928900
50,1.056700
60,0.987700
70,0.936200
80,0.971200
90,0.913100
100,0.972800


#### Code samples AFTER training

In [30]:
trained_samples = {}

for prompt in tqdm(prompts, desc="Generating trained samples", total=len(prompts)):
    trained_samples[prompt] = generate_code(model, prompt, max_length=100)

print_samples(prompts, trained_samples)
console.print("[bold green]Так гораздо лучше![/bold green]")


Generating trained samples:   0%|          | 0/10 [00:00<?, ?it/s]

                                 Baseline Samples (Before Training)                                  
┏━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Prompt   ┃ Sample                                                                                 ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ ''       │    1 from collections import defaultdict                                               │
│          │    2 import json                                                                       │
│          │    3 from typing import Dict                                                           │
│          │    4                                                                                   │
│          │    5 from django.conf import settings                                                  │
│          │    6 from django.core.exceptions import ImproperlyConfigured                           │
│          │    7                                                                                   │
│          │    8 from zerver.lib.queue import queue_json_publish                                   │
│          │    9 from zerver.lib.management import ZulipBaseCommand, CommandError                  │
│          │   10 from zerver.lib.onboarding import get_realm_subdomains                            │
│          │   11 from zerver.models import                                                         │
├──────────┼────────────────────────────────────────────────────────────────────────────────────────┤
│ 'import' │    1 import collections                                                                │
│          │    2 import datetime                                                                   │
│          │    3 import locale                                                                     │
│          │    4 import re                                                                         │
│          │    5 import time                                                                       │
│          │    6                                                                                   │
│          │    7 from django.conf import settings                                                  │
│          │    8 from django.core.exceptions import ImproperlyConfigured                           │
│          │    9 from django.utils.dates import MONTHS, MONTH_RE, DATETIME_RE, DATETIME_FULL_RE, \ │
│          │   10     PATTERN                                                                       │
│          │   11 from django.utils.functional import cached_property                               │
│          │   12 from django.utils.translation import                                              │
├──────────┼────────────────────────────────────────────────────────────────────────────────────────┤
│ 'from'   │    1 from django.test import TestCase                                                  │
│          │    2 from django.test.client import RequestFactory                                     │
│          │    3 from django.core.urlresolvers import reverse                                      │
│          │    4                                                                                   │
│          │    5 from django.contrib.auth.models import User                                       │
│          │    6 from django.contrib.auth.models import Group                                      │
│          │    7 from django.contrib.auth.models import GroupUser                                  │
│ 

Так гораздо лучше!

#### Compare results side-by-side

In [31]:
# Print all samples as a single panel with 3 columns: Prompt, Sample_left, Sample_right
def print_samples_sidebyside(prompts, samples_left, samples_right):
    table = Table(title="Baseline Comparison", show_header=True, header_style="bold green", show_lines=True)
    table.add_column("Prompt", style="cyan", overflow="crop")
    table.add_column("BEFORE", style="green", overflow="crop")
    table.add_column("AFTER", style="green", overflow="crop")

    for prompt in prompts:
        table.add_row(
            repr(prompt),
            Syntax(samples_left.get(prompt, ""), "python", theme="monokai", line_numbers=True),
            Syntax(samples_right.get(prompt, ""), "python", theme="monokai", line_numbers=True)
        )

    console.print(table)

print_samples_sidebyside(prompts, baseline_samples, trained_samples)

                                                Baseline Comparison                                                
┏━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Prompt   ┃ BEFORE                                           ┃ AFTER                                             ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ ''       │   1 ▲▼▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲ │    1 from collections import defaultdict          │
│          │                                                  │    2 import json                                  │
│          │                                                  │    3 from typing import Dict                      │
│          │                                                  │    4                                              │
│          │                                                  │    5 from django.conf import settings             │
│          │                                                  │    6 from django.core.exceptions import Improperl │
│          │                                                  │    7                                              │
│          │                                                  │    8 from zerver.lib.queue import queue_json_publ │
│          │                                                  │    9 from zerver.lib.management import ZulipBaseC │
│          │                                                  │   10 from zerver.lib.onboarding import get_realm_ │
│          │                                                  │   11 from zerver.models import                    │
├──────────┼──────────────────────────────────────────────────┼───────────────────────────────────────────────────┤
│ 'import' │    1 import Foundation                           │    1 import collections                           │
│          │    2                                             │    2 import datetime                              │
│          │    3 internal protocol NSObject {                │    3 import locale                                │
│          │    4                                             │    4 import re                                    │
│          │    5 }                                           │    5 import time                                  │
│          │    6                                             │    6                                              │
│          │    7 extension NSObject: NSCopying {             │    7 from django.conf import settings             │
│          │    8                                             │    8 from django.core.exceptions import Improperl │
│          │    9     internal func copy() -> NSObject {      │    9 from django.utils.dates import MONTHS, MONTH │
│          │   10         return self                         │   10     PATTERN                                  │
│          │   11     }                                       │   11 from django.utils.functional import cached_p │
│          │   12 }                                           │   12 from django.utils.

#### Test with more complex prompts

In [32]:
complex_prompts = [
    "def fibonacci",
    "import numpy as np",
    "class DataLoader",
    "if __name__ == '__main__':",
    "for i in range"
]

complex_samples = {}
for prompt in tqdm(complex_prompts, desc="Generating complex samples", total=len(complex_prompts)):
    complex_samples[prompt] = generate_code(model, prompt, max_length=200)

print_samples(complex_prompts, complex_samples)


Generating complex samples:   0%|          | 0/5 [00:00<?, ?it/s]

                                        Baseline Samples (Before Training)                                         
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Prompt                       ┃ Sample                                                                           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 'def fibonacci'              │    1 def fibonacci(n):                                                           │
│                              │    2     a, b = 0, 1                                                             │
│                              │    3     while b < n:                                                            │
│                              │    4         a, b = b, a+b                                                       │
│                              │    5     return a                                                                │
│                              │    6                                                                             │
│                              │    7                                                                             │
│                              │    8 def main():                                                                 │
│                              │    9     for _ in range(int(input())):                                           │
│                              │   10         assert fibonacci(int(input())) == int(input())                      │
│                              │   11                                                                             │
│                              │   12 if __name__ == '__main__':                                                  │
│                              │   13     main()                                                                  │
│                              │   14                                                                             │
├──────────────────────────────┼──────────────────────────────────────────────────────────────────────────────────┤
│ 'import numpy as np'         │    1 import numpy as np                                                          │
│                              │    2 from collections import namedtuple                                          │
│                              │    3 from abc import ABCMeta, abstractmethod                                     │
│                              │    4 from ast import literal_eval as ast_literal_eval                            │
│                              │    5                                                                             │
│                              │    6 from .base import (BaseEstimator, TransformerMixin, _LossEstimatorMixin)    │
│                              │    7 from .base import _LossClassifierMixin                                      │
│                              │    8 from ..utils import (as_float_array, check_array, check_X_y,                │
│                              │    9                      check_sample_weight)                                   │
│                              │   10 from ..utils.extmath import safe_sparse_dot                                 │
│                              │   11 from ..utils.validation import check_is_fitted                              │
│                              │   12                                                                             │
│                              │   13                                                                             │
│                              │   14 def _loss_and_metric_loss_doc_kwargs(loss, metric_operator):                │
│                              │   15     """                                                                     │
│                              │   16     Parameters    

In [ ]:
# # This template helps to compare generated code samples in pretty table form
# # feel free to present your work in other forms

# from IPython.display import HTML, display
# table_template = """<table style="border:1px solid black" >
#   <tr>
#     <th style="text-align: center; border:1px solid black">PROMPT</th>
#     <th style="text-align: center; border:1px solid black">BEFORE</th>
#     <th style="text-align: center; border:1px solid black">AFTER</th>
#   </tr>
# {}
# </table>"""

# row_template = '''  <tr>
#     <td style="width:20%; border:1px solid black"><pre align="left">`{}`</pre></td>
#     <td style="width:40%; border:1px solid black"><pre align="left">{}</pre></td>
#     <td style="width:40%; border:1px solid black"><pre align="left">{}</pre></td>
#   </tr>'''

# rows = []

# for prompt in prompts:
#     # replace placeholders in the format() arguments
#     rows.append(row_template.format(prompt, "BEFORE FINETUNING", "TO BE GENERATED AFTER FINETUNING"))

# display(HTML(table_template.format('\n'.join(rows))))

If you reach this: congratulations! you've completed everything in this practice session.

If you want to dig deeper, try to implement prompt-tuning (for bonus points!).
You can read more about prompt tuning variants in paper [1](https://arxiv.org/abs/2104.08691) or paper [2](https://arxiv.org/abs/2101.00190). Both versions can be implemented by passing trainable prompts as `model.forward(..., past_key_values=your_prompts)`.



### Read more

* How post-training quantization works: https://arxiv.org/abs/2208.07339
* An overview of running large models: https://huggingface.co/docs/accelerate/package_reference/big_modeling
* A general library for different adapter types: https://adapterhub.ml/


### [extra info] Running other models.

This notebook's code can run with other models of similar size, such as [Falcon-7B](https://huggingface.co/tiiuae/falcon-7b), [OPT-6.7B](https://huggingface.co/facebook/opt-6.7b) or [BLOOM-7.1B](https://huggingface.co/bigscience/bloom-7b1). However, they will require minor code tweaks:
1. change the model name in `AutoModelForCausalLM.from_pretrained()` __and__ `AutoTokenizer`
2. In the prompt tuning code, change `model.model.embed_tokens` to refer to the target model's word embeddings. Simply `print(model)` to navigate to them.
3. Change code to add Lora layers - specifically where you what the transformer block components, since those components now have different names.